# 01 · Exploración de los datos
**Proyecto:** Búsqueda de Empleo Sin Experiencia · **Asignatura:** Minería de Datos · **Docente:** Paola Andrea Ortiz  
**Integrantes:** Bryan Samuel Vélez Velásquez · Jhon Jairo Melo Largo

**Pregunta del proyecto:** ¿Qué características de los jóvenes y de las vacantes aumentan la probabilidad de que alguien sin experiencia consiga su primer empleo en Colombia, para orientar a qué sectores y formación dirigirlos?

**Objetivo de este notebook:** conocer la base de datos antes de limpiarla y analizarla: qué columnas tiene, de qué tipo son, cuántos datos faltan y qué patrones se ven a primera vista.

> ⚠️ Por ahora se usa `datos_simulados_empleo_sin_experiencia.csv`, una base **simulada** (columna `fuente = SIMULADO`) para practicar. Se reemplazará por los datos reales de la GEIH (DANE) y del Servicio Público de Empleo.

## 1. Librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Mostrar tablas completas (Colab y Jupyter recortan filas, columnas y textos largos)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
import os

ARCHIVO = "datos_simulados_empleo_sin_experiencia.csv"
URL_GITHUB = "https://raw.githubusercontent.com/samuelvelez1003/Busqueda-Empleo/main/data/" + ARCHIVO

for ruta in ["../data/" + ARCHIVO, "data/" + ARCHIVO, ARCHIVO]:
    if os.path.exists(ruta):
        df = pd.read_csv(ruta)
        print("Datos cargados desde:", ruta)
        break
else:
    df = pd.read_csv(URL_GITHUB)
    print("Datos cargados desde GitHub")

df.head()

## 3. Tamaño y tipos de datos

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

In [ ]:
df.info()

**Diccionario rápido de variables**

| Variable | Descripción |
|---|---|
| `id_persona` | Identificador de la persona |
| `anio`, `mes` | Periodo del registro (2025) |
| `edad`, `sexo`, `ciudad` | Datos básicos de la persona |
| `nivel_educativo`, `area_formacion` | Formación alcanzada |
| `tuvo_practica`, `cursos_cortos` | Preparación adicional |
| `meses_buscando` | Tiempo buscando empleo |
| `sector_aspirado` | Sector donde busca trabajo |
| `consiguio_empleo` | **Variable objetivo**: 1 = sí, 0 = no |
| `tipo_contrato`, `empleo_formal`, `salario_mensual_cop` | Solo para quienes consiguieron empleo |

## 4. Datos faltantes

In [ ]:
faltantes = df.isna().sum().to_frame("faltantes")
faltantes["porcentaje"] = (faltantes["faltantes"] / len(df) * 100).round(1)
faltantes[faltantes["faltantes"] > 0].sort_values("faltantes", ascending=False)

**Interpretación:** `tipo_contrato`, `empleo_formal` y `salario_mensual_cop` están vacíos cuando la persona **no** consiguió empleo, así que esos vacíos son esperados. Los faltantes de `meses_buscando`, `cursos_cortos` y `area_formacion` sí se deben tratar en el notebook de limpieza.

## 5. Estadísticas descriptivas

In [ ]:
df.describe().round(1)

In [ ]:
df.describe(include="object")

## 6. Variable objetivo: ¿consiguió empleo?

In [ ]:
tasa = df["consiguio_empleo"].mean() * 100
print(f"Porcentaje que consiguió empleo: {tasa:.1f}%")

df["consiguio_empleo"].map({1: "Sí", 0: "No"}).value_counts().plot(kind="bar", color=["#1F3A5F", "#9FB3C8"])
plt.title("¿Consiguió empleo?")
plt.ylabel("Personas")
plt.xticks(rotation=0)
plt.show()

## 7. Primeros patrones
Se calcula el **porcentaje de jóvenes que consiguió empleo** en cada grupo. Esto permite una primera revisión de la hipótesis: *la formación técnica/tecnológica y el sector influyen en conseguir el primer empleo.*

In [ ]:
def tasa_empleo(columna):
    tabla = (df.groupby(columna)["consiguio_empleo"]
               .agg(personas="count", tasa_empleo="mean")
               .sort_values("tasa_empleo", ascending=False))
    tabla["tasa_empleo"] = (tabla["tasa_empleo"] * 100).round(1)
    return tabla

def grafica_tasa(columna, titulo):
    t = tasa_empleo(columna)["tasa_empleo"].sort_values()
    t.plot(kind="barh", color="#1F3A5F")
    plt.title(titulo)
    plt.xlabel("% que consiguió empleo")
    plt.ylabel("")
    plt.show()

### 7.1 Por nivel educativo

In [ ]:
display(tasa_empleo("nivel_educativo"))
grafica_tasa("nivel_educativo", "Consiguió empleo según nivel educativo")

### 7.2 Según si tuvo práctica

In [ ]:
display(tasa_empleo("tuvo_practica"))
grafica_tasa("tuvo_practica", "Consiguió empleo según si tuvo práctica")

### 7.3 Por sector al que aspira

In [ ]:
display(tasa_empleo("sector_aspirado"))
grafica_tasa("sector_aspirado", "Consiguió empleo según sector")

### 7.4 Por ciudad

In [ ]:
display(tasa_empleo("ciudad"))
grafica_tasa("ciudad", "Consiguió empleo según ciudad")

### 7.5 Edad

In [ ]:
df["edad"].plot(kind="hist", bins=11, color="#1F3A5F", edgecolor="white")
plt.title("Distribución de edad")
plt.xlabel("Edad")
plt.show()

### 7.6 Salario de quienes consiguieron empleo

In [ ]:
ocupados = df[df["consiguio_empleo"] == 1]
(ocupados.groupby("nivel_educativo")["salario_mensual_cop"]
         .median().sort_values().plot(kind="barh", color="#1F3A5F"))
plt.title("Salario mensual mediano (COP) según nivel educativo")
plt.xlabel("COP")
plt.ylabel("")
plt.show()

print("Porcentaje con empleo formal:", round((ocupados["empleo_formal"] == "Sí").mean() * 100, 1), "%")